# ДЗ-12

## 1. постановка задачи

матрица $A$ размера $8 \times 8$ строится из sha-256 хэша строки Бабаева А.В. 501290. шестьдесят четыре шестнадцатеричные цифры хэша записываются по строкам слева направо сверху вниз и переводятся в десятичную систему.

для матрицы $M = A A^{T}$ требуется построить приближение матрицей ранга 3.

матрица $M = A A^{T}$ симметрична и положительно полуопределена. для симметричной матрицы существует спектральное разложение $M = \sum_{k=1}^{n} \lambda_k v_k v_k^{T}$, где $\lambda_k$ это собственные числа, а $v_k$ это ортонормированные собственные векторы. наилучшее в норме фробениуса приближение матрицей ранга 3 получается удержанием трёх слагаемых с наибольшими по модулю собственными числами.

## 2. построение матрицы

In [1]:
import numpy as np
import hashlib

In [2]:
s = 'Бабаева А.В. 501290'
h = hashlib.sha256(s.encode('utf-8')).hexdigest()
print('sha-256:', h)

A = np.array([int(c, 16) for c in h]).reshape(8, 8).astype(float)
M = A @ A.T
print('матрица M = A A^T:')
print(M.astype(int))

sha-256: 06f1c694eff244523e0f74e7e1235b7cb1e12b9281fc49bde8b693dec1a9fcf9
матрица M = A A^T:
[[ 539  442  361  276  396  496  518  588]
 [ 442  711  410  370  482  509  634  552]
 [ 361  410  740  362  260  527  599  611]
 [ 276  370  362  549  404  531  581  636]
 [ 396  482  260  404  529  543  518  597]
 [ 496  509  527  531  543  821  745  805]
 [ 518  634  599  581  518  745  872  832]
 [ 588  552  611  636  597  805  832 1001]]


## 3. спектральное разложение

находим собственные числа и собственные векторы симметричной матрицы $M$ и упорядочиваем их по убыванию собственных чисел.

In [3]:
eigvals, eigvecs = np.linalg.eigh(M)

poryadok = np.argsort(eigvals)[::-1]
eigvals = eigvals[poryadok]
eigvecs = eigvecs[:, poryadok]

print('собственные числа по убыванию:')
for val in eigvals:
    print(f'  {val:.4f}')

собственные числа по убыванию:
  4498.6962
  429.7743
  353.7023
  236.6719
  116.4645
  87.5261
  24.4962
  14.6685


## 4. приближение рангом 3

матрица ранга 3 строится как сумма трёх слагаемых спектрального разложения с наибольшими собственными числами:

$$M_3 = \lambda_1 v_1 v_1^{T} + \lambda_2 v_2 v_2^{T} + \lambda_3 v_3 v_3^{T}.$$

In [4]:
M3 = np.zeros((8, 8))
for k in range(3):
    lam = eigvals[k]
    v = eigvecs[:, k].reshape(8, 1)
    M3 = M3 + lam * (v @ v.T)

print('приближение M3:')
print(np.round(M3, 1))
print()
print('ранг M3:', np.linalg.matrix_rank(M3))

приближение M3:
[[415.8 511.5 372.7 331.9 409.1 489.3 546.5 542.2]
 [511.5 664.5 405.2 323.7 482.1 527.4 617.3 574.4]
 [372.7 405.2 726.1 359.1 240.  527.1 619.9 611.9]
 [331.9 323.7 359.1 481.9 411.  574.7 558.3 658. ]
 [409.1 482.1 240.  411.  496.  538.6 546.8 596.2]
 [489.3 527.4 527.1 574.7 538.6 735.7 756.  835.2]
 [546.5 617.3 619.9 558.3 546.8 756.  811.2 855.1]
 [542.2 574.4 611.9 658.  596.2 835.2 855.1 951. ]]

ранг M3: 3


## 5. оценка точности приближения

в качестве меры близости используется норма фробениуса матрицы разности $\| M - M_3 \|_F$. для симметричной матрицы эта величина равна корню из суммы квадратов отброшенных собственных чисел.

In [5]:
oshibka = np.linalg.norm(M - M3, 'fro')
teoriya = np.sqrt(np.sum(eigvals[3:]**2))

print('норма фробениуса ||M - M3||_F =', round(oshibka, 4))
print('корень из суммы квадратов отброшенных чисел =', round(teoriya, 4))
print()
norma_M = np.linalg.norm(M, 'fro')
print('относительная ошибка ||M - M3||_F / ||M||_F =', round(oshibka / norma_M, 6))

норма фробениуса ||M - M3||_F = 279.3808
корень из суммы квадратов отброшенных чисел = 279.3808

относительная ошибка ||M - M3||_F / ||M||_F = 0.061516


## 6. выводы

приближение $M_3$ построено как сумма трёх слагаемых спектрального разложения матрицы $M = A A^{T}$ с наибольшими собственными числами. ранг полученной матрицы равен 3.

норма фробениуса разности $\| M - M_3 \|_F$ приблизительно равна $279.38$ и совпадает с корнем из суммы квадратов пяти отброшенных собственных чисел. относительная ошибка приближения мала, ибо наибольшее собственное число значительно превосходит остальные, и основная доля нормы матрицы сосредоточена в первых слагаемых разложения.